# Pricing Explorer — prices by vehicle class, mileage band, and term

Shows how prices move across risk classes, mileage, and term length. Assumes you've run `01_getting_started.ipynb` once so the URLs and schema are familiar.

**License:** CC BY 4.0. Cite as: *OptimalCover (2026). VSC Pricing Dataset (v3.6.7).*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/Optimal-Cover/vsc-pricing-data/main/data/v3.6.7"
rates = pd.read_csv(f"{BASE_URL}/vsc_rates.csv")

# Only the canonical $100 deductible is stored; that's what we'll analyze.
assert (rates['deductible'] == 100).all(), "Expected only $100 rows in vsc_rates"
print(f"{len(rates)} rows across {rates['vehicle_class'].nunique()} classes")

## 1. Average reference price by vehicle class

Class A is the most reliable (cheapest to cover). Class D is the least (most expensive to cover).

In [ ]:
by_class = rates.groupby('vehicle_class')['price'].agg(['mean', 'min', 'max']).round(0)
by_class

In [ ]:
ax = by_class['mean'].plot(kind='bar', color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'])
ax.set_title('Average reference price by vehicle class ($100 deductible)')
ax.set_ylabel('Price (USD)')
ax.set_xlabel('Vehicle class')
plt.tight_layout()
plt.show()

## 2. Price curve by term length

Longer terms cost more. How much more depends on the vehicle class.

In [ ]:
# Average across mileage bands and low/high bands, grouped by class x term
term_curve = (
    rates.groupby(['vehicle_class', 'term_months'])['price']
         .mean()
         .unstack('vehicle_class')
         .round(0)
)
term_curve

In [ ]:
ax = term_curve.plot(marker='o')
ax.set_title('Average price vs term length (by vehicle class)')
ax.set_xlabel('Term (months)')
ax.set_ylabel('Average price (USD)')
ax.legend(title='Class')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Reference range spread (HIGH − LOW)

The spread between `REFERENCE_LOW` and `REFERENCE_HIGH` is how much typical dealer markup inflates the price over a competitively priced offer.

In [ ]:
ranges = rates.pivot_table(
    index=['vehicle_class', 'mileage_band', 'term_months'],
    columns='band',
    values='price',
    aggfunc='first',
).reset_index()
ranges['spread'] = ranges['REFERENCE_HIGH'] - ranges['REFERENCE_LOW']
ranges['spread_pct'] = (ranges['spread'] / ranges['REFERENCE_LOW'] * 100).round(1)

ranges.groupby('vehicle_class')[['spread', 'spread_pct']].mean().round(1)

---

**Takeaway:** Reference ranges are wide because real-world VSC quotes vary widely. If you're shopping for one, landing near `REFERENCE_LOW` means you got a good deal; landing near or above `REFERENCE_HIGH` means you paid dealer markup.